In [0]:
# ============================================================
# Pipeline : covid19_medallion_pipeline
# Notebook : 04_silver/05_silver_medications
# Purpose  : Medications + Quarantine - Actual rule discovered from
#            profiling: 808 rows where STOP < START.
# ============================================================

import dlt
from pyspark.sql import functions as F

TEMPORAL_VALID_RULE_DATE = "stop_date IS NULL OR stop_date >= start_date"


@dlt.table(name="covid19_lakehouse.silver.medications", comment="Medications passing DQ rules (STOP >= START)")
@dlt.expect_all_or_drop({
    "valid_patient_id": "patient_id IS NOT NULL",
    "valid_encounter_id": "encounter_id IS NOT NULL",
})
@dlt.expect_or_drop("valid_stop_after_start", TEMPORAL_VALID_RULE_DATE)
def silver_medications():
    df = dlt.read_stream("covid19_lakehouse.bronze.medications")
    return (
        df.select(
            F.col("PATIENT").alias("patient_id"),
            F.col("ENCOUNTER").alias("encounter_id"),
            F.col("PAYER").alias("payer_id"),
            F.col("CODE").alias("medication_code"),
            F.col("DESCRIPTION").alias("medication_description"),
            F.to_date("START").alias("start_date"),
            F.to_date("STOP").alias("stop_date"),
            F.when(F.col("STOP").isNull(), True).otherwise(False).alias("is_active"),
            F.col("DISPENSES").cast("int").alias("dispenses"),
            F.col("BASE_COST").cast("decimal(10,2)").alias("base_cost"),
            F.col("TOTALCOST").cast("decimal(10,2)").alias("total_cost"),
            F.col("PAYER_COVERAGE").cast("decimal(10,2)").alias("payer_coverage"),
            F.col("REASONCODE").alias("reason_code"),
            F.col("REASONDESCRIPTION").alias("reason_description"),
            F.col("_ingested_at"),
        )
    )


@dlt.table(
    name="covid19_lakehouse.quarantine.medications",
    comment="Medications rejected by Silver DQ rules (null FK or STOP < START)",
)
def quarantine_medications():
    df = dlt.read_stream("covid19_lakehouse.bronze.medications")
    return (
        df.filter(
            F.col("PATIENT").isNull()
            | F.col("ENCOUNTER").isNull()
            | (F.col("STOP").isNotNull() & (F.to_date("STOP") < F.to_date("START")))
        )
        .select(
            F.lit("medications").alias("source_table"),
            F.lit("null_fk / stop_before_start").alias("rule_violated"),
            F.col("PATIENT").alias("patient_id"),
            F.col("ENCOUNTER").alias("encounter_id"),
            F.col("START").alias("raw_start"),
            F.col("STOP").alias("raw_stop"),
            F.current_timestamp().alias("quarantined_at"),
        )
    )